# ASO offline (v2) - self-contained, without deep-significance

Recomputes the **ASO comparison** (Almost Stochastic Order) for the seed-paired
comparison - **without the PyPI package** (which can no longer be installed in
current Colab) and **without** touching the locked test set. It is based on the
already stored `test_eval/test_fold_predictions_*.csv` from `test_evaluation_v2`;
from these the five fold macro-F1 per architecture are reconstructed. **No model
is loaded and nothing is re-scored.**

The **violation ratio** is exactly the quantity from Del Barrio et al. (2018) /
Dror et al. (2019) on which the package is also based. In addition, a
conservative upper bound `eps_min` is estimated by bootstrap. Needs only
numpy/pandas/scipy (preinstalled in Colab) - no external installation.

Reported per pair:
- **eps_hat (violation ratio, point)**: 0 = the better architecture fully
  dominates the weaker one (no overlap); 0.5 = tie; 1 = reversed.
- **eps_min (bootstrap upper bound, 95%)**: conservative, confidence-corrected
  eps - small (< 0.5, clearly < 0.2) allows the claim 'better'.
- **Vargha-Delaney A12**: superiority probability, 0.5 = no effect.
- **paired t-test p** as a cross-check.

**Output:** `test_eval/significance_aso.csv`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os, json, itertools
# --- locate the project root -------------------------------------------
# No hardcoded Drive path: take KUSA_ROOT if it is set, otherwise the first
# candidate that actually contains config.py. Works in Colab and locally.
import os, sys
_CANDIDATES = [
    os.environ.get("KUSA_ROOT", ""),
    "/content/drive/MyDrive/v2_heldout",
    "/content/drive/MyDrive/google_colab/kusa/v2_heldout",
    os.getcwd(),
    os.path.dirname(os.getcwd()),
]
V2_ROOT = next((p for p in _CANDIDATES
                if p and os.path.isfile(os.path.join(p, "config.py"))), None)
assert V2_ROOT, ("config.py not found - set KUSA_ROOT to the v2_heldout "
                 "directory, e.g. os.environ['KUSA_ROOT'] = '/content/drive/MyDrive/v2_heldout'")
sys.path.insert(0, V2_ROOT)
print("project root:", V2_ROOT)
from config import *

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from scipy.stats import ttest_rel

print("Self-contained ASO implementation - no external installation required.")

In [ ]:
# Reconstruct per-architecture fold macro-F1 from the stored test predictions.
FOLD_F1 = {}
ref_labels = None
for v in VARIANTS:
    path = os.path.join(TEST_EVAL, f"test_fold_predictions_{v}.csv")
    assert os.path.exists(path), (
        f"missing: {path}\n-> test_evaluation_v2 must have run.")
    df = pd.read_csv(path, encoding="utf-8")
    y = df["label"].values
    if ref_labels is None:
        ref_labels = y
    else:
        assert np.array_equal(y, ref_labels), (
            f"labels/order in {v} differ from the other variants.")
    f1s = np.array([f1_score(y, df[f"pred_fold{k}"].values, average="macro")
                    for k in range(1, N_FOLDS + 1)])
    FOLD_F1[v] = f1s
    print(f"{v:20s} Folds " + ", ".join(f"{x:.4f}" for x in f1s)
          + f"  | mean {f1s.mean():.4f} +/- {f1s.std(ddof=1):.4f}")

print("\nReconstructed from stored fold predictions - test set NOT re-scored.")

In [ ]:
# --- ASO core (self-contained) -------------------------------------------
# Violation ratio after Del Barrio et al. (2018) / Dror et al. (2019):
# fraction of the squared Wasserstein distance pointing toward "b > a" zeigt.
# eps_hat(a, b) small  =>  a dominates b (a has the larger values).
def violation_ratio(a, b, dt=0.005):
    ps = np.arange(dt / 2, 1.0, dt)
    qa = np.quantile(a, ps)
    qb = np.quantile(b, ps)
    diff = qb - qa                       # > 0 where b is larger (violation)
    w2 = float(np.sum(diff ** 2))
    if w2 == 0:
        return 0.5
    return float(np.sum(np.clip(diff, 0, None) ** 2) / w2)


# Conservative upper bound: bootstrap over the actual fold values (with
# replacement), then the (confidence) quantile of the bootstrap distribution as
# a one-sided upper bound. Important: with only 5 folds one MUST resample with n=5 -
# drawing many points from the quantile function would obscure the small sample
# and yield overconfident (too small) eps_min.
def aso_epsilon(a, b, n_boot=2000, confidence=0.95, seed=ASO_SEED):
    a = np.asarray(a, float); b = np.asarray(b, float)
    eps_hat = violation_ratio(a, b)
    rng = np.random.default_rng(seed)
    n = len(a)
    boot = np.empty(n_boot)
    for i in range(n_boot):
        boot[i] = violation_ratio(rng.choice(a, n, replace=True),
                                  rng.choice(b, n, replace=True))
    eps_min = float(np.clip(np.quantile(boot, confidence), 0, 1))
    return eps_hat, eps_min


def vargha_a12(a, b):
    """P(a > b) + 0.5 P(a = b): 0.5 = no effect, 1.0 = a always better."""
    a, b = np.asarray(a), np.asarray(b)
    gt = sum(x > y for x in a for y in b)
    eq = sum(x == y for x in a for y in b)
    return (gt + 0.5 * eq) / (len(a) * len(b))


rows = []
for x, y in itertools.combinations(VARIANTS, 2):
    fx, fy = FOLD_F1[x], FOLD_F1[y]
    better, worse = (x, y) if fx.mean() >= fy.mean() else (y, x)
    fb, fw = FOLD_F1[better], FOLD_F1[worse]
    eps_hat, eps_min = aso_epsilon(fb, fw)      # better dominiert worse?
    t = ttest_rel(fb, fw)
    row = {
        "better": better, "worse": worse,
        "mean_diff": round(float(fb.mean() - fw.mean()), 4),
        "eps_hat": round(eps_hat, 4),
        "eps_min_95": round(eps_min, 4),
        "A12_better_over_worse": round(vargha_a12(fb, fw), 3),
        "paired_t_p": round(float(t.pvalue), 4),
        "dominates": bool(eps_min < 0.5),
    }
    rows.append(row)

    print(f"\n{better}  >  {worse}   (Delta {row['mean_diff']:+.4f})")
    print(f"  eps_hat (violation ratio) : {row['eps_hat']}   (0 = no overlap)")
    print(f"  eps_min (bootstrap 95%)   : {row['eps_min_95']}   (< 0.5 => dominates)")
    print(f"  Vargha-Delaney A12        : {row['A12_better_over_worse']}   (0.5 = no effect)")
    print(f"  paired t-test p        : {row['paired_t_p']}")
    print("  -> " + ("stochastic dominance confirmed" if row["dominates"]
                     else "not distinguishable"))

aso_df = pd.DataFrame(rows)
aso_df.to_csv(os.path.join(TEST_EVAL, "significance_aso.csv"),
              index=False, encoding="utf-8")
print("\n" + aso_df.to_string(index=False))
print("\nsaved:", os.path.join(TEST_EVAL, "significance_aso.csv"))

## Reading

| Quantity | better dominates | indistinguishable |
|---|---|---|
| **eps_hat** | 0 | toward 0.5 |
| **eps_min (95%)** | < 0.5 (clearly < 0.2) | around/above 0.5 |
| **A12** | near 1.0 | around 0.5 |
| **t-test p** | small (< 0.05) | large |

Expectation from the results so far: `dual_view`/`gated` **dominate** the
`baseline` (non-overlapping fold distributions -> eps_hat = eps_min = 0,
A12 = 1.0), while `dual_view` vs `gated` is **indistinguishable**
(eps_min -> 1.0, A12 ~ 0.6, t-test p ~ 0.7).

With only five folds the bootstrap upper bound is conservative; the paired
t-test remains the primary number, eps and A12 are the distribution-free
cross-check.

**Cite:** Del Barrio, Cuesta-Albertos & Matran (2018); Dror, Shlomov &
Reichart (2019), *Deep Dominance - How to Properly Compare Deep Neural Models*,
ACL. The violation ratio is their core quantity; `eps_min` here is a bootstrap
upper bound instead of the package's normal approximation.
